## Map showing the direction of the ship

In [1]:
from ipyleaflet import Map, Marker, AwesomeIcon, Popup, Rectangle
from ipywidgets import HTML
import math

# If using actint modules, import constants
from actint.tools.utils.important_locations import MARITIME_REGIONS, CONTINENTS, OCEANS

# Create map centered somewhere reasonable and set zoom
map1 = Map(center=(20.0, 0.0), zoom=2)

# Optional helper for text popup at center

def create_bounding_box_on_map(map_obj, bounds, color='blue', fill_opacity=0.15, weight=2, name=None):
    """Add one or two rectangles for bounding boxes and optional text popup label."""
    required_keys = {'lat_min', 'lat_max', 'lon_min', 'lon_max'}
    if not required_keys.issubset(bounds):
        raise ValueError(f"bounds must include {required_keys}")

    lat_min = float(bounds['lat_min'])
    lat_max = float(bounds['lat_max'])
    lon_min = float(bounds['lon_min'])
    lon_max = float(bounds['lon_max'])

    # Normalize lat range
    if lat_min > lat_max:
        lat_min, lat_max = lat_max, lat_min

    rectangles = []
    if lon_min <= lon_max:
        rectangles.append((lat_min, lon_min, lat_max, lon_max))
    else:
        rectangles.append((lat_min, lon_min, lat_max, 180))
        rectangles.append((lat_min, -180, lat_max, lon_max))

    added = []
    for rlat_min, rlon_min, rlat_max, rlon_max in rectangles:
        rect = Rectangle(
            bounds=[
                (rlat_min, rlon_min),
                (rlat_min, rlon_max),
                (rlat_max, rlon_max),
                (rlat_max, rlon_min),
            ],
            color=color,
            weight=weight,
            fill_color=color,
            fill_opacity=fill_opacity,
        )
        map_obj.add_layer(rect)
        added.append(rect)

    if name is not None:
        center_lat = (lat_min + lat_max) / 2
        if lon_min <= lon_max:
            center_lon = (lon_min + lon_max) / 2
        else:
            center_lon = ((lon_min + 180) % 360 - 180 + (lon_max - 180) % 360 + 180) / 2
        popup = Popup(
            location=(center_lat, center_lon),
            child=HTML(f"<b>{name}</b>"),
            close_button=False,
            auto_close=False,
            close_on_click=False,
        )
        map_obj.add(popup)

    return added if len(added) > 1 else added[0]

for key, value in MARITIME_REGIONS.items():
    create_bounding_box_on_map(map1, value['bounds'], color='cyan', fill_opacity=0.08, weight=1, name=key)

for key, value in CONTINENTS.items():
    create_bounding_box_on_map(map1, value['bounds'], color='green', fill_opacity=0.05, weight=2, name=key)

for key, value in OCEANS.items():
    create_bounding_box_on_map(map1, value['bounds'], color='blue', fill_opacity=0.04, weight=2, name=key)

map1

/home/daxtonb/.conda/envs/actint/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


Map(center=[20.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_…

## Some sampole vectors and locations for where the ship is:

## Utility Functions

In [2]:
DEGREE_THRESHOLD = 15

# current_position = (15, -153) #Near Hawaii going toward North America/(United States)
# direction_vector = (0, 10)

# current_position = (28, -59) # In the Atlantic off the coast of the United States going toward the Gulf of America
# direction_vector = (-10, -2.5)

current_position = (40, 17) # In the middle of the Mediter
direction_vector = (-6, 9)

In [3]:
import math

def vector_to_latlon_degrees(lat_diff, lon_diff):
    """
    Converts a (lat, lon) vector into a compass bearing from -180 to 180.
    0 = North, 90 = East, 180/-180 = South, -90 = West.
    """
    radians = math.atan2(lon_diff, lat_diff)
    degrees = math.degrees(radians)

    return degrees

# Examples using your vectorize output:
print(f"North (1, 0):   {vector_to_latlon_degrees(1, 0)}°") 
print(f"East (0, 1):    {vector_to_latlon_degrees(0, 1)}°")
print(f"South (-1, 0):  {vector_to_latlon_degrees(-1, 0)}°")
print(f"West (0, -1):   {vector_to_latlon_degrees(0, -1)}°")

North (1, 0):   0.0°
East (0, 1):    90.0°
South (-1, 0):  180.0°
West (0, -1):   -90.0°


## Ship position and direction display

In [4]:
map2 = Map(center=(current_position[0], current_position[1]), zoom=10)

# Add first and last markers
current_pos_marker = Marker(location=(current_position[0], current_position[1]))
map2.add(current_pos_marker)

(dy, dx) = direction_vector



start_lat = current_position[0]
start_lon = current_position[1]

end_lat = start_lat + dy
end_lon = start_lon + dx

from ipyleaflet import Polyline

vector_line = Polyline(
    locations=[(start_lat, start_lon), (end_lat, end_lon)],
    color="red",
    weight=3
)

map2.add(vector_line)

map2

Map(center=[40, 17], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

## The code which determines where in the world the ship is going

### Get Possible Destinations

In [5]:


from geographiclib.geodesic import Geodesic

# Define the WGS84 ellipsoid
GEOD = Geodesic.WGS84

def project_point(lat, lon, bearing, distance_km):
    # geographiclib uses meters, so we multiply km by 1000
    # Direct returns a dictionary with 'lat2' and 'lon2'
    result = GEOD.Direct(lat, lon, bearing, distance_km * 1000)
    return result['lat2'], result['lon2']

def view_approximation(lat, lon, bearing, map_obj, degree_threshold, distance_km=10000):
    # Upper bound line
    points_up = []
    for i in range(1, 51):
        dist = distance_km * (i / 50)
        points_up.append(project_point(lat, lon, bearing + degree_threshold, dist))
    
    line_up = Polyline(
        locations=points_up,
        color="red",
        weight=1
    )
    map_obj.add(line_up)

    # Lower bound line
    points_down = []
    for i in range(1, 51):
        dist = distance_km * (i / 50)
        points_down.append(project_point(lat, lon, bearing - degree_threshold, dist))
    
    line_down = Polyline(
        locations=points_down,
        color="blue",
        weight=1 # Assuming you wanted the second line thinner or default
    )
    map_obj.add(line_down)

# Usage
DEGREE_THRESHOLD = 5 # Example value
view_approximation(current_position[0], current_position[1], vector_to_latlon_degrees(direction_vector[0], direction_vector[1]), map2, DEGREE_THRESHOLD)
map2

Map(center=[40, 17], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

### The following code gives a simple sample heading calculation, it should be slightly higher than 180 degrees becuase is goes straight down from the current position and then a little left.

In [6]:
from geographiclib.geodesic import Geodesic

# Define the WGS84 ellipsoid (the standard for GPS)
geod = Geodesic.WGS84

# Inverse calculation: find the path BETWEEN two points
# (lat1, lon1, lat2, lon2)
path = geod.Inverse(current_position[0], current_position[1], -85, current_position[1]-1)

# 'azi1' is the initial heading (Forward Azimuth)
# 'azi2' is the arrival heading (Back Azimuth)
initial_heading = path['azi1']

#The sample heading should be slightly less than 180 degrees
print(f"Sample Heading: {initial_heading:.2f}°")

#Go through all the potential points using this function and return the ones that are within DEGREE_THRESHOLD for being a possible destination location

Sample Heading: -179.89°


In [8]:
from actint.tools.utils.important_locations import *
from actint.tools.utils.distance_calculation import calculate_bearing, haversine_distance_nm

map3 = Map(center=(20.0, 0.0), zoom=0)

def in_area(current_position, heading, min_distance, max_distance, area, step_size=200):
    bounds = area["bounds"]
    
    for distance in range(int(min_distance), int(max_distance), int(step_size)):
        (lat, lon) = project_point(current_position[0], current_position[1], heading, distance)
        if bounds['lat_min'] < lat < bounds['lat_max'] and bounds['lon_min'] < lon < bounds['lon_max']:
            return distance
    return None



heading_degrees = vector_to_latlon_degrees(direction_vector[0], direction_vector[1])
current_locations = get_locations(current_position)
current_location_names = {loc[0] for loc in current_locations}

print(f"skipping {current_locations} becuase you are already there")
matches = []
ave_distances = []

def going_to_locations(current_position, locations, skip_locations, heading_degrees):
    area_hits = []
    for name, vars in locations.items():
        if name in skip_locations:
            continue

        bounds = vars["bounds"]
        lat_min = bounds['lat_min']; lon_min = bounds['lon_min']; lat_max = bounds['lat_max']; lon_max = bounds['lon_max']

        coordinates = [(lat_min, lon_min), (lat_min, lon_max), (lat_max, lon_min), (lat_max, lon_max), ((lat_min + lat_max)/2, (lon_min + lon_max)/2)] # tests the center area as well to reduce the chance of missing the object

        lengths_to_area = []
        for coordinate in coordinates:
            path = geod.Inverse(current_position[0], current_position[1], coordinate[0], coordinate[1])
            lengths_to_area.append(path['s12'])
        
        min_length = min(lengths_to_area)
        
        distance1 = in_area(current_position, heading_degrees + DEGREE_THRESHOLD, min_length/1000, float(15000), vars)
        distance2 = in_area(current_position, heading_degrees - DEGREE_THRESHOLD, min_length/1000, float(15000), vars)

        if(distance1 and not distance2):
            area_hits.append((name, distance1))
        if(distance2 and not distance1):
            area_hits.append((name, distance2))
        if(distance1 and distance2):
            area_hits.append((name, min(distance1, distance2)))   # Return the shortest distance away from the area detected


    return area_hits


ocean_hits = going_to_locations(current_position, OCEANS, current_location_names, heading_degrees)
continents_hits = going_to_locations(current_position, CONTINENTS, current_location_names, heading_degrees)
maritime_hits = going_to_locations(current_position, MARITIME_REGIONS, current_location_names, heading_degrees)

# print(ocean_hits)
# print(continents_hits
print(maritime_hits)

for key, value in CONTINENTS.items():
    create_bounding_box_on_map(map3, value['bounds'], color='green', fill_opacity=0.05, weight=2, name=key)

view_approximation(current_position[0], current_position[1], vector_to_latlon_degrees(direction_vector[0], direction_vector[1]), map3, DEGREE_THRESHOLD, distance_km = 15000)

map3

skipping [('Mediterranean Sea', 672), ('Europe', 2537)] becuase you are already there
[('Arabian Sea', 4090), ('Red Sea', 1957), ('Indian Ocean', 1742)]


Map(center=[20.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_…

## Highlighted regions that the ship route on the map (15 degree error threshold)

In [9]:
map4 = Map(center=(20.0, 0.0), zoom=0)

def highlight_matches(hits_list, source_dict, highlight_color):
    for name, distance in hits_list:

        bounds = source_dict[name]['bounds']
        
        create_bounding_box_on_map(
            map4, 
            bounds, 
            color=highlight_color, 
            fill_opacity=0.3,  # Higher opacity for hits
            weight=3,          # Thicker border
            name=f"HIT: {name}"
        )

# 2. Run the highlights for each category
# Using distinct colors to differentiate the types of regions
highlight_matches(continents_hits, CONTINENTS, 'gold')
highlight_matches(ocean_hits, OCEANS, 'cyan')
highlight_matches(maritime_hits, MARITIME_REGIONS, 'magenta')


view_approximation(current_position[0], current_position[1], heading_degrees, map4, 0, 15000)

map4

Map(center=[20.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_…

## You can now take the results and give it to the LLM

In [12]:
continents_hits.sort(key=lambda x: x[1])    # Sort the items based on which one is closer
ocean_hits.sort(key=lambda x: x[1])
maritime_hits.sort(key=lambda x: x[1])

print(continents_hits)
print(ocean_hits)
print(maritime_hits)

#put the results into some kind of a string and feed it into an LLM:

print(f"The ship is in {current_locations[0][0]} headed toward the {maritime_hits[0][0]}, {ocean_hits[0][0]}, and {continents_hits[0][0]}")

[('Africa', 3077), ('Asia', 3993), ('Austrailia', 13136)]
[('Pacific Ocean (Western)', 11696)]
[('Indian Ocean', 1742), ('Red Sea', 1957), ('Arabian Sea', 4090)]
The ship is in Mediterranean Sea headed toward the Indian Ocean, Pacific Ocean (Western), and Africa
